In [ ]:
# Silver Layer - RetailMax Data Pipeline

Este notebook limpia y transforma los datos de Bronze a Silver.

In [ ]:
## 1. Instalar librerías y cargar configuración

In [ ]:
%pip install pandas python-dotenv azure-storage-file-datalake -q

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from azure.storage.filedatalake import DataLakeServiceClient
from io import BytesIO
from datetime import datetime
import hashlib

# Cargar configuración
load_dotenv()

AZURE_STORAGE_ACCOUNT = os.getenv("AZURE_STORAGE_ACCOUNT")
AZURE_STORAGE_KEY = os.getenv("AZURE_STORAGE_KEY")

In [ ]:
## 2. Conectar a ADLS y leer datos de Bronze

In [ ]:
# Conectar a ADLS
service_client = DataLakeServiceClient(
    account_url=f"https://{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=AZURE_STORAGE_KEY
)
bronze_fs = service_client.get_file_system_client("bronze")
silver_fs = service_client.get_file_system_client("silver")

# Leer todos los archivos Parquet de Bronze
dataframes = {}
for path in bronze_fs.get_paths():
    if path.name.endswith(".parquet"):
        table_name = path.name.split("/")[1].replace(".parquet", "")
        file_client = bronze_fs.get_file_client(path.name)
        df = pd.read_parquet(BytesIO(file_client.download_file().readall()))
        dataframes[table_name] = df
        print(f"✓ {table_name}: {len(df):,} registros")

In [ ]:
## 3. Función auxiliar para subir a ADLS

In [ ]:
def upload_to_adls(df, table_name):
    """Sube dataframe a ADLS en formato Parquet con overwrite para idempotencia"""
    temp_file = f"temp_{table_name}.parquet"
    df.to_parquet(temp_file, index=False)
    
    dir_client = silver_fs.get_directory_client(table_name.upper())
    try:
        dir_client.create_directory()
    except:
        pass
    
    file_client = dir_client.get_file_client(f"{table_name}.parquet")
    with open(temp_file, "rb") as data:
        file_client.upload_data(data, overwrite=True)  # overwrite=True asegura idempotencia
    
    os.remove(temp_file)
    print(f"✓ {table_name.upper()} cargado en Silver (overwrite=True)")

In [ ]:
## 4. Transformaciones Silver

In [ ]:
# Asignar dataframes
articulos = dataframes["mstr_articulos"]
proveedores = dataframes["mstr_proveedores"]
tiendas = dataframes["mstr_tiendas"]
miembros = dataframes["crm_miembros"]
ventas = dataframes["fact_ventas"]
stock = dataframes["inv_stock_diario"]
devoluciones = dataframes["fact_devoluciones"]

# Inicializar tabla de errores del pipeline
pipeline_errors = pd.DataFrame(columns=['timestamp', 'tabla', 'error', 'registro_id'])

def add_error(errors_df, table_name, error_msg, record_id=None):
    """Agrega un error a la tabla de errores"""
    new_error = pd.DataFrame([{
        'timestamp': datetime.now(),
        'tabla': table_name,
        'error': error_msg,
        'registro_id': record_id
    }])
    return pd.concat([errors_df, new_error], ignore_index=True)

0       False
1       False
2       False
3       False
4       False
        ...  
4995    False
4996    False
4997    False
4998    False
4999    False
Length: 5000, dtype: bool

In [ ]:
## 5. Generar reporte de calidad

In [ ]:
# Generar reporte simple de calidad
start_time = datetime.now()

quality_report = pd.DataFrame([
    {'tabla': 'dim_productos', 'originales': len(articulos), 'limpios': len(dim_productos), 'rechazados': len(articulos) - len(dim_productos)},
    {'tabla': 'dim_tiendas', 'originales': len(tiendas), 'limpios': len(dim_tiendas), 'rechazados': len(tiendas) - len(dim_tiendas)},
    {'tabla': 'dim_clientes', 'originales': len(miembros), 'limpios': len(dim_clientes), 'rechazados': len(miembros) - len(dim_clientes)},
    {'tabla': 'fact_ventas', 'originales': len(ventas), 'limpios': len(fact_ventas), 'rechazados': len(ventas) - len(fact_ventas)},
    {'tabla': 'fact_inventario', 'originales': len(stock), 'limpios': len(fact_inventario), 'rechazados': len(stock) - len(fact_inventario)},
    {'tabla': 'fact_devoluciones', 'originales': len(devoluciones), 'limpios': len(fact_devoluciones), 'rechazados': len(devoluciones) - len(fact_devoluciones)},
])
quality_report['tasa_rechazo_pct'] = (quality_report['rechazados'] / quality_report['originales'] * 100).round(2)

# Agregar métricas de ejecución
execution_report = pd.DataFrame([{
    'timestamp': datetime.now(),
    'duracion_segundos': (datetime.now() - start_time).total_seconds(),
    'tablas_procesadas': len(quality_report),
    'total_registros_originales': quality_report['originales'].sum(),
    'total_registros_limpios': quality_report['limpios'].sum(),
    'total_duplicados_eliminados': quality_report['rechazados'].sum(),
    'total_errores_pipeline': len(pipeline_errors)
}])

print("Reporte de calidad de datos:")
print(quality_report)
print("\nReporte de ejecución:")
print(execution_report)

In [ ]:
## 6. Cargar datos a Silver

In [ ]:
# Cargar todas las tablas
upload_to_adls(dim_productos, 'dim_productos')
upload_to_adls(dim_tiendas, 'dim_tiendas')
upload_to_adls(dim_clientes, 'dim_clientes')
upload_to_adls(fact_ventas, 'fact_ventas')
upload_to_adls(fact_inventario, 'fact_inventario')
upload_to_adls(fact_devoluciones, 'fact_devoluciones')
upload_to_adls(quality_report, 'quality_report')
upload_to_adls(execution_report, 'execution_report')
upload_to_adls(pipeline_errors, 'pipeline_errors')

In [ ]:
## 7. Resumen

In [ ]:
print("=" * 50)
print("✓ Proceso Silver completado")
print(f"✓ {len(quality_report)} tablas procesadas")
print("=" * 50)

In [ ]:
# Join con proveedores
dim_productos = articulos.merge(
    proveedores[['id_proveedor', 'nombre_proveedor', 'pais', 'calificacion']],
    left_on='mstr_proveedores_id_proveedor',
    right_on='id_proveedor',
    how='left'
)

# Eliminar duplicados
dim_productos = dim_productos.drop_duplicates(subset=['id_articulo'])

# Eliminar registros con campos obligatorios nulos
dim_productos = dim_productos.dropna(subset=[
    'id_articulo', 
    'nombre_producto', 
    'categoria',
    'precio'
])

# Estandarizar tipos de datos
dim_productos['precio'] = pd.to_numeric(dim_productos['precio'], errors='coerce')
dim_productos['calificacion'] = pd.to_numeric(dim_productos['calificacion'], errors='coerce')

print(f"dim_productos: {len(dim_productos):,} registros (después de limpieza)")

In [ ]:
## Transformación: dim_tiendas

In [ ]:
# Estandarizar tipo_tienda a catálogo controlado
tipo_tienda_mapping = {
    'hipermercado': 'HIPERMERCADO',
    'supermercado': 'SUPERMERCADO',
    'tienda_conveniencia': 'CONVENIENCIA',
    'hiper': 'HIPERMERCADO',
    'super': 'SUPERMERCADO',
    'conveniencia': 'CONVENIENCIA'
}

dim_tiendas = tiendas.copy()
dim_tiendas['tipo_tienda'] = dim_tiendas['tipo_tienda'].str.lower().map(tipo_tienda_mapping).fillna('OTRO')

# Eliminar duplicados
dim_tiendas = dim_tiendas.drop_duplicates(subset=['id_tienda'])

# Eliminar registros con campos obligatorios nulos
dim_tiendas = dim_tiendas.dropna(subset=['id_tienda', 'tipo_tienda', 'pais'])

print(f"dim_tiendas: {len(dim_tiendas):,} registros (después de limpieza)")

In [ ]:
## Transformación: dim_clientes

In [ ]:
# Calcular antigüedad en días desde fec_registro
dim_clientes = miembros.copy()
dim_clientes['fec_registro'] = pd.to_datetime(dim_clientes['fec_registro'], errors='coerce')
dim_clientes['antiguedad_dias'] = (datetime.now() - dim_clientes['fec_registro']).dt.days

# Imputar rango_edad nulo con la mediana del canal preferido
if 'rango_edad' in dim_clientes.columns:
    median_age_by_channel = dim_clientes.groupby('canal_pref')['rango_edad'].transform(lambda x: x.mode()[0] if not x.mode().empty else '26-35')
    dim_clientes['rango_edad'] = dim_clientes['rango_edad'].fillna(median_age_by_channel)

# Estandarizar género a M, F o No informado
genero_mapping = {
    'masculino': 'M',
    'm': 'M',
    'male': 'M',
    'femenino': 'F',
    'f': 'F',
    'female': 'F'
}
dim_clientes['genero'] = dim_clientes['genero'].str.lower().map(genero_mapping).fillna('NO_INFORMADO')

# Enmascarar datos sensibles (PII)
dim_clientes['id_miembro_hash'] = dim_clientes['id_miembro'].apply(hash_pii)
dim_clientes['email_hash'] = dim_clientes.get('email', pd.Series([None]*len(dim_clientes))).apply(hash_pii)

# Eliminar duplicados
dim_clientes = dim_clientes.drop_duplicates(subset=['id_miembro'])

# Eliminar registros con campos obligatorios nulos
dim_clientes = dim_clientes.dropna(subset=['id_miembro'])

print(f"dim_clientes: {len(dim_clientes):,} registros (después de limpieza y enmascaramiento)")

In [ ]:
## Transformación: fact_ventas

In [ ]:
# Calcular vr_venta_neto = qty_vendida x precio_unitario - descuento
fact_ventas = ventas.copy()
fact_ventas['fecha_hora'] = pd.to_datetime(fact_ventas['fecha_hora'], errors='coerce')
fact_ventas['vr_venta_neto'] = fact_ventas['precio'] - fact_ventas.get('descuento', 0)

# Validar id_miembro contra dim_clientes
fact_ventas_valid, fact_ventas_errors = validate_referential_integrity(
    fact_ventas, 
    dim_clientes, 
    'id_miembro', 
    'id_miembro',
    'fact_ventas'
)

# Asignar cliente anónimo para registros sin miembro válido
fact_ventas_valid['id_miembro'] = fact_ventas_valid['id_miembro'].fillna('ANONIMO')

# Agregar indicador de venta con descuento
fact_ventas_valid['con_descuento'] = fact_ventas_valid.get('descuento', 0) > 0

# Eliminar duplicados
fact_ventas_valid = fact_ventas_valid.drop_duplicates(subset=['id_venta'])

# Eliminar registros con campos obligatorios nulos
fact_ventas_valid = fact_ventas_valid.dropna(subset=['id_venta', 'id_articulo', 'id_tienda'])

print(f"fact_ventas: {len(fact_ventas_valid):,} registros válidos")
print(f"fact_ventas: {len(fact_ventas_errors):,} registros con errores de integridad")

In [ ]:
## Transformación: fact_inventario

In [ ]:
# Validar integridad referencial
fact_inventario_valid, fact_inventario_errors = validate_referential_integrity(
    stock,
    dim_productos,
    'id_articulo',
    'id_articulo',
    'fact_inventario'
)

fact_inventario_valid, fact_inventario_errors_tiendas = validate_referential_integrity(
    fact_inventario_valid,
    dim_tiendas,
    'id_tienda',
    'id_tienda',
    'fact_inventario'
)

# Concatenar errores
fact_inventario_errors = pd.concat([fact_inventario_errors, fact_inventario_errors_tiendas])

# Calcular cobertura_dias (requiere cálculo de promedio de consumo - simplificado aquí)
fact_inventario_valid['cobertura_dias'] = fact_inventario_valid['stock_fisico'] / (fact_inventario_valid['stock_fisico'] / 14 + 1)

# Flag alerta_quiebre cuando cobertura_dias < 7
fact_inventario_valid['alerta_quiebre'] = fact_inventario_valid['cobertura_dias'] < 7

# Calcular diferencia frente a stock_minimo_config
fact_inventario_valid['diferencia_stock_min'] = fact_inventario_valid['stock_fisico'] - fact_inventario_valid['stock_minimo']

# Eliminar duplicados
fact_inventario_valid = fact_inventario_valid.drop_duplicates(subset=['id_tienda', 'id_articulo'])

print(f"fact_inventario: {len(fact_inventario_valid):,} registros válidos")
print(f"fact_inventario: {len(fact_inventario_errors):,} registros con errores de integridad")

In [ ]:
## Transformación: fact_devoluciones

In [ ]:
# Join con la venta origen para obtener precio original
fact_devoluciones = devoluciones.merge(
    fact_ventas_valid[['id_venta', 'precio']],
    on='id_venta',
    how='left'
)

# Estandarizar motivo_cod a descripción legible
motivo_mapping = {
    'DEF': 'Producto defectuoso',
    'TAM': 'Talla o tamaño incorrecto',
    'ARR': 'Arrepentimiento de compra',
    'VEN': 'Producto vencido',
    'ERR': 'Error en el pedido'
}
if 'motivo' in fact_devoluciones.columns:
    fact_devoluciones['motivo_descripcion'] = fact_devoluciones['motivo'].map(motivo_mapping).fillna('Otro')

# Validar integridad referencial
fact_devoluciones_valid, fact_devoluciones_errors = validate_referential_integrity(
    fact_devoluciones,
    dim_productos,
    'id_articulo',
    'id_articulo',
    'fact_devoluciones'
)

# Eliminar duplicados
fact_devoluciones_valid = fact_devoluciones_valid.drop_duplicates(subset=['id_venta'])

print(f"fact_devoluciones: {len(fact_devoluciones_valid):,} registros válidos")
print(f"fact_devoluciones: {len(fact_devoluciones_errors):,} registros con errores de integridad")

In [ ]:
## Consolidación de errores del pipeline

In [ ]:
# Consolidar todos los errores de integridad referencial
pipeline_errors = pd.concat([
    fact_ventas_errors,
    fact_inventario_errors,
    fact_devoluciones_errors
], ignore_index=True)

print(f"Total de errores de integridad referencial: {len(pipeline_errors):,}")

if len(pipeline_errors) > 0:
    # Guardar tabla de errores
    upload_to_adls(pipeline_errors, silver_fs, 'pipeline_errors')
else:
    print("No se encontraron errores de integridad referencial")

In [ ]:
## Generación de reporte de calidad de datos

In [ ]:
def generate_quality_report(original_dfs, transformed_dfs):
    """Genera reporte de calidad de datos con % de nulos y registros rechazados"""
    report_data = []
    
    for table_name in original_dfs.keys():
        original_count = len(original_dfs[table_name])
        transformed_count = len(transformed_dfs.get(table_name, pd.DataFrame()))
        rejected_count = original_count - transformed_count
        rejection_rate = (rejected_count / original_count * 100) if original_count > 0 else 0
        
        # Calcular % de nulos por columna
        null_percentages = {}
        for col in original_dfs[table_name].columns:
            null_count = original_dfs[table_name][col].isna().sum()
            null_pct = (null_count / original_count * 100) if original_count > 0 else 0
            null_percentages[col] = round(null_pct, 2)
        
        report_data.append({
            'tabla': table_name,
            'registros_originales': original_count,
            'registros_transformados': transformed_count,
            'registros_rechazados': rejected_count,
            'tasa_rechazo_pct': round(rejection_rate, 2),
            'porcentaje_nulos_por_columna': null_percentages,
            'porcentaje_conforme_pct': round(100 - rejection_rate, 2)
        })
    
    return pd.DataFrame(report_data)

# Dataframes transformados
transformed_dfs = {
    'mstr_articulos': dim_productos,
    'mstr_tiendas': dim_tiendas,
    'crm_miembros': dim_clientes,
    'fact_ventas': fact_ventas_valid,
    'inv_stock_diario': fact_inventario_valid,
    'fact_devoluciones': fact_devoluciones_valid
}

# Generar reporte
quality_report = generate_quality_report(dataframes, transformed_dfs)
print("Reporte de calidad de datos:")
print(quality_report[['tabla', 'registros_originales', 'registros_transformados', 'registros_rechazados', 'tasa_rechazo_pct', 'porcentaje_conforme_pct']])

In [ ]:
## Carga de datos transformados a ADLS Silver

In [ ]:
# Cargar dimensiones
upload_to_adls(dim_productos, silver_fs, 'dim_productos')
upload_to_adls(dim_tiendas, silver_fs, 'dim_tiendas')
upload_to_adls(dim_clientes, silver_fs, 'dim_clientes')

# Cargar hechos
upload_to_adls(fact_ventas_valid, silver_fs, 'fact_ventas')
upload_to_adls(fact_inventario_valid, silver_fs, 'fact_inventario')
upload_to_adls(fact_devoluciones_valid, silver_fs, 'fact_devoluciones')

# Cargar reporte de calidad
upload_to_adls(quality_report, silver_fs, 'quality_report')

In [ ]:
## Resumen de ejecución

In [ ]:
print("=" * 60)
print("Proceso Silver finalizado exitosamente.")
print(f"Total de tablas procesadas: {len(transformed_dfs)}")
print(f"Total de errores de integridad: {len(pipeline_errors):,}")
print(f"Reporte de calidad generado y cargado")
print("=" * 60)